# SafeQuery on Colab (Run 7): is the serving wrapper part of the result?
**Lexsi Labs hackathon, Agentic Fine-tuning track**: AgentTune + AuditKIT on **Tiny Aya Global**.

**Why this run.** An organizer pointed out that the stop-token / `trim_turn` configuration should be part of the reported system identity: our own 16.7% → 0% invalid-call fix shows that output handling changes measured behaviour without any training. Run 7 measures that directly.

**What it does. No training.**
1. Downloads the **published** Run 5 adapters from the Hugging Face Hub, the exact files a reader would use, and records their commit hashes.
2. Evaluates base, SFT and GRPO on the 111 held-out tasks under **two serving wrappers**:
   - `safequery`: the settings behind every reported number (greedy, stop on every end-of-turn token and on `"Tool result for"`, then `trim_turn()`).
   - `plain`: a bare `generate()` call with the model's default stopping, no stop string and no trimming.
3. Every `metrics.json` now holds a **`serving`** block (stop tokens, stop strings, post-processing, decoding, library versions) and **`wrapper_stats`**, the raw behaviour *before* trimming. Every raw turn is saved to `raw_turns.jsonl`.
4. Writes `results_run7/WRAPPERS.md`: model alone vs model + wrapper, plus a check that the `safequery` rows still equal Run 5.

**Runtime:** A100. Install 5-15 min · each evaluation ~25-40 min (plain turns can run to the token limit) · 6 evaluations ≈ 3-4 h. Set `RUN_GRPO = False` to halve it.
Results go to **`results_run7/`** and **`MyDrive/safequery_run7`** after every evaluation. Every step skips work that's already done.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import sys; print(sys.version)  # AgentTune needs Python >= 3.12

## 0 - Get the code from GitHub (needs the Run 7 wrapper code)

In [ ]:
import os
PROJ = "/content/LLMHackathon2026"
REPO_URL = "https://github.com/siva2k16/LLMHackathon2026"
REPO_BRANCH = "run7"   # the Run 7 code lives on this branch until the results are in
if not os.path.exists(f"{PROJ}/scripts/build_data.py"):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {PROJ}
else:
    !git -C {PROJ} pull -q origin {REPO_BRANCH}
%cd {PROJ}
!git log --oneline -1
ok = "def wrapper_stats" in open("safequery/policies.py").read() and "--wrapper" in open("scripts/eval_model.py").read()
print("Run 7 wrapper code present:", ok)
assert ok, f"Push the Run 7 code to the {REPO_BRANCH} branch on GitHub first"

## 1 - Install AgentTune and AuditKIT (unmodified, from Lexsi Labs)
5-15 minutes. Red **dependency conflict** warnings are harmless.
**After it installs, use Runtime -> Restart session**, then continue from the next cell.

In [ ]:
!git clone -q https://github.com/Lexsi-Labs/AgentTune.git /content/AgentTune 2>/dev/null || echo "AgentTune already cloned"
!git clone -q https://github.com/Lexsi-Labs/AuditKIT.git /content/AuditKIT 2>/dev/null || echo "AuditKIT already cloned"
import subprocess
if subprocess.run(["python", "-c", "import agenttune, auditkit"], capture_output=True).returncode == 0:
    print("AgentTune + AuditKIT already installed - skipping")
else:
    !pip install -q --only-binary=pandas -e /content/AgentTune -e /content/AuditKIT
    print("Installed. Now: Runtime -> Restart session, then run the next cells.")

In [ ]:
import sys
for p in ["/content/AgentTune/src", "/content/AuditKIT/src", "/content/AuditKIT"]:
    if p not in sys.path: sys.path.insert(0, p)
import torch, transformers, peft, agenttune, auditkit
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| transformers", transformers.__version__, "| peft", peft.__version__)

## 1b - Hugging Face login
The base model is **gated**: accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global first.
A **read** token is enough (nothing is uploaded in this run). Never paste tokens into code cells.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Config - **re-run this cell after every restart or disconnect**

In [ ]:
import os, glob, shutil, hashlib, json, time, torch
PROJ = "/content/LLMHackathon2026"
%cd {PROJ}

MODEL     = "CohereLabs/tiny-aya-global"
RES       = "results_run7"
RUN_GRPO  = True     # False = base + SFT only (about half the time)
LIMIT     = None     # None = all 111 test tasks
LOAD_4BIT = False
HF_ADAPTERS = {"sft":  "Siva2022/safequery-tiny-aya-global-sft-lora",
               "grpo": "Siva2022/safequery-tiny-aya-global-grpo-lora"}

LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
Q_ARG     = "--load-4bit" if LOAD_4BIT else ""
DRIVE_DIR = "/content/drive/MyDrive/safequery_run7"
GPU_NAME  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
os.makedirs(RES, exist_ok=True)

RUN5_MD5 = {
    "tasks_train.jsonl": "1297c43b4683d016f9529620a41bb0b9",
    "tasks_test.jsonl":  "aa5cd9fac7e2f137f52974e1d6a0643a",
    "sft_train.jsonl":   "ab308961f042e5cefe54205c1eef60df",
    "grpo_train.jsonl":  "e38fa26dcbf4e7c5a8815d4c9eaf99c0",
}

def _mount():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

def _md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None

def _done(name):
    p = f"{RES}/{name}/metrics.json"
    return os.path.exists(p) and "wrapper_stats" in json.load(open(p))

def save_to_drive():
    _mount(); os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copytree(RES, f"{DRIVE_DIR}/{RES}", dirs_exist_ok=True)
    print("saved ->", DRIVE_DIR)

TIMES = {}
def timed(label, t0):
    TIMES[label] = round((time.time() - t0) / 60, 1); print(f"[time] {label}: {TIMES[label]} min")

print("GPU:", GPU_NAME, "| cwd", os.getcwd())

## R - *Only after a disconnect*: restore finished evaluations from Drive

In [ ]:
RESTORE = False   # set True after a disconnect
if RESTORE:
    _mount()
    if os.path.exists(f"{DRIVE_DIR}/{RES}"):
        shutil.copytree(f"{DRIVE_DIR}/{RES}", RES, dirs_exist_ok=True); print("restored", sorted(os.listdir(RES)))
else:
    print("RESTORE is False - fresh run")

## 2 - Build the database and tasks, check they match Run 5

In [ ]:
%cd {PROJ}
!python scripts/build_data.py
ok = True
for f, want in RUN5_MD5.items():
    got = _md5(f"data/{f}")
    print(f"{f:20s}", "MATCH" if got == want else f"DIFFERENT ({got})")
    ok &= got == want
print("all match Run 5:", ok)

## 3 - Download the published Run 5 adapters from the Hub
These are the files a reader would download. Their commit hashes are saved to `results_run7/adapters.json`.

In [ ]:
%cd {PROJ}
from huggingface_hub import HfApi, snapshot_download
api = HfApi(); ADAPTER_INFO = {}
for name, repo in HF_ADAPTERS.items():
    if name == "grpo" and not RUN_GRPO:
        continue
    path = snapshot_download(repo, local_dir=f"adapters/{name}")
    sha = api.model_info(repo).sha
    assert os.path.exists(f"adapters/{name}/adapter_model.safetensors"), f"no weights in {repo}"
    ADAPTER_INFO[name] = {"repo": repo, "commit": sha, "local": f"adapters/{name}"}
    print(name, repo, sha[:10])
json.dump(ADAPTER_INFO, open(f"{RES}/adapters.json", "w"), indent=2)

## 4 - Merge the SFT adapter (GRPO's starting model)
The GRPO adapter was trained on top of the merged SFT model, so it is evaluated on that base.

In [ ]:
%cd {PROJ}
if RUN_GRPO and not os.path.exists("runs/sft_merged/config.json"):
    t0 = time.time()
    !python scripts/merge_adapter.py --model {MODEL} --adapter adapters/sft --out runs/sft_merged
    timed("merge", t0)
else:
    print("merge not needed or already done")

## 5 - Evaluate every model under both wrappers
`safequery` = the reported settings · `plain` = bare `generate()`. Saved to Drive after each evaluation.

In [ ]:
%cd {PROJ}
RUNS = [("base", MODEL, None), ("sft", MODEL, "adapters/sft")]
if RUN_GRPO:
    RUNS.append(("grpo", "runs/sft_merged", "adapters/grpo"))
for model_name, model, adapter in RUNS:
    for wrapper in ("safequery", "plain"):
        name = model_name if wrapper == "safequery" else f"{model_name}_plain"
        if _done(name):
            print(name, "already done - skipping"); continue
        ad = f"--adapter {adapter}" if adapter else ""
        t0 = time.time()
        !python scripts/eval_model.py --name {name} --model {model} {ad} --wrapper {wrapper} --results {RES} {LIMIT_ARG} {Q_ARG}
        timed(f"{name} eval", t0)
        save_to_drive()

## 6 - Model alone vs model + wrapper
`raw_overrun_rate` = share of turns where the model wrote past its own turn, measured on the raw output **before** any trimming, the same way for both wrappers.
The check at the bottom compares the `safequery` rows with the Run 5 numbers in `results/`: the logging change must not change behaviour.

In [ ]:
%cd {PROJ}
from IPython.display import Markdown, display
COLS = [("task_success_rate", "Task success"), ("answer_accuracy", "Answer acc."), ("refusal_accuracy", "Refusals"),
        ("injection_resistance", "Injection res."), ("unsafe_attempt_rate", "Unsafe attempts"),
        ("invalid_call_rate", "Invalid calls")]
WCOLS = [("raw_overrun_rate", "Raw overrun"), ("invented_tool_result_rate", "Invented tool result"),
         ("hit_token_limit_rate", "Hit token limit"), ("avg_new_tokens", "Avg new tokens")]
pct = lambda v: f"{100*v:.1f}%"
rows = ["| Run | Wrapper | " + " | ".join(c for _, c in COLS + WCOLS) + " | tok/s |",
        "|---|---|" + "---|" * (len(COLS) + len(WCOLS) + 1)]
for model_name, _, _ in RUNS:
    for wrapper in ("safequery", "plain"):
        name = model_name if wrapper == "safequery" else f"{model_name}_plain"
        p = f"{RES}/{name}/metrics.json"
        if not os.path.exists(p):
            continue
        m = json.load(open(p)); M, W = m["metrics"], m["wrapper_stats"]
        rows.append(f"| {model_name} | {wrapper} | " + " | ".join(pct(M[k]) for k, _ in COLS) + " | "
                    + " | ".join(pct(W[k]) if k != "avg_new_tokens" else str(W[k]) for k, _ in WCOLS)
                    + f" | {m['throughput']['tokens_per_sec']} |")
table = "\n".join(rows)

check = ["", "**Check: `safequery` rows vs Run 5 (`results/`)**", ""]
for model_name, _, _ in RUNS:
    p5, p7 = f"results/{model_name}/metrics.json", f"{RES}/{model_name}/metrics.json"
    if os.path.exists(p5) and os.path.exists(p7):
        m5, m7 = json.load(open(p5))["metrics"], json.load(open(p7))["metrics"]
        diff = [k for k, _ in COLS if abs(m5[k] - m7[k]) > 1e-9]
        check.append(f"- {model_name}: " + ("identical to Run 5" if not diff else "DIFFERENT on " + ", ".join(diff)))

srv = json.load(open(f"{RES}/sft/metrics.json"))["serving"] if os.path.exists(f"{RES}/sft/metrics.json") else {}
contract = ["", "**Serving contract recorded with the SFT result**", "", "```json", json.dumps(srv, indent=2), "```"]
report = "## Run 7: model alone vs model + serving wrapper\n\n" + table + "\n" + "\n".join(check + contract)
open(f"{RES}/WRAPPERS.md", "w", encoding="utf-8").write(report)
display(Markdown(report))

In [ ]:
# One raw turn where the model ran past its own turn (plain wrapper), as an example
%cd {PROJ}
for name in ["sft_plain", "grpo_plain", "base_plain"]:
    p = f"{RES}/{name}/raw_turns.jsonl"
    if os.path.exists(p):
        ex = next((json.loads(l) for l in open(p, encoding="utf-8") if json.loads(l)["overrun"]), None)
        if ex:
            print(f"--- {name} · {ex['task_id']} · {ex['new_tokens']} new tokens ---")
            print(ex["raw"][:900]); break
else:
    print("no overrun found in the plain runs")

## 7 - Save and download

In [ ]:
%cd {PROJ}
json.dump({"gpu": GPU_NAME, "minutes": TIMES}, open(f"{RES}/run7_timing.json", "w"), indent=2)
save_to_drive()
!zip -r -q /content/safequery_results_run7.zip {RES}
!ls -lh /content/safequery_results_run7.zip
from google.colab import files
files.download("/content/safequery_results_run7.zip")